In [0]:
import pandas as pd
import numpy as np

df = spark.table("workspace.gold.training_dataset_features_fifa").toPandas()

print(df.shape)
display(df.head())

In [0]:
df.columns.tolist()

In [0]:
#filtre datos modernos

#Para el primer modelo, usemos partidos desde el año 2000. Así evitamos que partidos muy antiguos afecten demasiado.

df_model = df[df["year"] >= 2000].copy()

print("Filas para modelado:", df_model.shape)
print(df_model["result"].value_counts())

In [0]:
#definimos las variables del modelo 

features = [
    "tournament_weight",
    "is_world_cup",
    "is_world_cup_qualifier",
    "is_friendly",
    "is_neutral",
    
    "home_recent_games",
    "away_recent_games",
    
    "home_recent_win_rate",
    "away_recent_win_rate",
    "home_recent_draw_rate",
    "away_recent_draw_rate",
    "home_recent_loss_rate",
    "away_recent_loss_rate",
    
    "home_recent_avg_goals_for",
    "away_recent_avg_goals_for",
    "home_recent_avg_goals_against",
    "away_recent_avg_goals_against",
    "home_recent_avg_goal_diff",
    "away_recent_avg_goal_diff",
    
    "recent_win_rate_diff",
    "recent_draw_rate_diff",
    "recent_loss_rate_diff",
    "recent_avg_goals_for_diff",
    "recent_avg_goals_against_diff",
    "recent_avg_goal_diff_diff",
    "recent_games_diff",

    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "fifa_rank_diff",
    "fifa_points_diff"
]

target = "target"

df_model[features] = df_model[features].fillna(0)

X = df_model[features]
y = df_model[target]

print(X.shape)
print(y.value_counts())

In [0]:

#No hacemos split aleatorio. Entrenamos con el pasado y probamos con partidos más reciente
train = df_model[df_model["date"] < "2022-01-01"].copy()
test = df_model[df_model["date"] >= "2022-01-01"].copy()

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

logistic_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        multi_class="multinomial",
        class_weight="balanced"
    ))
])

logistic_model.fit(X_train, y_train)

log_pred = logistic_model.predict(X_test)
log_proba = logistic_model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, log_pred))
print("Log Loss:", log_loss(y_test, log_proba))
print(classification_report(y_test, log_pred))

In [0]:
import pandas as pd

labels = ["home_win", "draw", "away_win"]

cm = confusion_matrix(y_test, log_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real {label}" for label in labels],
    columns=[f"Pred {label}" for label in labels]
)

display(cm_df)

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Log Loss:", log_loss(y_test, rf_proba))
print(classification_report(y_test, rf_pred))

In [0]:
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance)

In [0]:
model_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_test, log_pred),
        "log_loss": log_loss(y_test, log_proba)
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test, rf_pred),
        "log_loss": log_loss(y_test, rf_proba)
    }
])

display(model_results)

In [0]:
model_results_spark = spark.createDataFrame(model_results)

model_results_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold.model_results_base")

print("Resultados de modelos base guardados en Gold.")